# Notebook 6 — Figuras Finales

Genera las 7 figuras del paper: distribución FSM, matriz de transición, clustering PCA, Z por cluster, boxplot Z2, pie modos cognitivos, heatmap Spearman.

In [ ]:
import os, numpy as np, random, pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from scipy.stats import spearmanr
SEED = 42
np.random.seed(SEED)
random.seed(SEED)
BASE_PATH = '.'
INPUTS  = os.path.join(BASE_PATH, 'data')
OUTPUTS = os.path.join(BASE_PATH, 'outputs')
ID_COL = 'email'; STATE_COL = 'state'; TIME_COL = 'timestamp'
print('Setup OK')


In [ ]:
traj_full      = pd.read_csv(f"{OUTPUTS}/trajectories_full.csv")
clusters       = pd.read_csv(f"{OUTPUTS}/clusters_k4.csv")
z_summary_prop = pd.read_csv(f"{OUTPUTS}/hmm_k4_hidden_states_summary.csv")
fsm_df         = pd.read_csv(f"{OUTPUTS}/fsm_v6_metrics_per_student.csv")
grades_df      = pd.read_csv(f"{INPUTS}/grades_export_anon.csv")
grades_df.columns = [c.strip().lower() for c in grades_df.columns]
Z_COLS = [c for c in z_summary_prop.columns if c.startswith('Z')]
GRADE_COL = next((c for c in ['nota_final','nota','final_grade'] if c in grades_df.columns), None)
print(f'GRADE_COL: {GRADE_COL} | Z_COLS: {Z_COLS}')


In [ ]:
# Figura 1 — Distribución global de estados FSM
fsm_dist = traj_full[STATE_COL].value_counts(normalize=True)
fig, ax = plt.subplots(figsize=(7,4))
fsm_dist.plot(kind='bar', ax=ax, color='steelblue', edgecolor='white')
ax.set_ylabel('Proportion'); ax.set_xlabel('FSM State')
ax.set_title('Figure 1. Distribution of FSM state proportions across students')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure1_FSM_distribution.png", dpi=300); plt.close()
print('Fig 1 guardada.')


In [ ]:
# Figura 2 — Matriz de transición
traj_s = traj_full.sort_values([ID_COL, TIME_COL]).copy()
traj_s['prev_state'] = traj_s.groupby(ID_COL)[STATE_COL].shift(1)
T = pd.crosstab(traj_s['prev_state'], traj_s[STATE_COL], normalize='index')
fig, ax = plt.subplots(figsize=(7,5))
sns.heatmap(T, annot=True, fmt='.2f', cmap='Blues', ax=ax, linewidths=0.5)
ax.set_title('Figure 2. FSM v6 transition matrix')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure2_FSM_transition_matrix.png", dpi=300); plt.close()
print('Fig 2 guardada.')


In [ ]:
# Figura 3 — PCA clustering
fsm_cols_pca = [c for c in ['NAV','REC','PRACT','QUIZ','EVAL'] if c in fsm_df.columns]
X_pca = fsm_df[fsm_cols_pca]
labels = clusters.set_index('email').loc[fsm_df['email'], 'cluster'].values
pca = PCA(n_components=2, random_state=SEED)
X_2d = pca.fit_transform(X_pca)
fig, ax = plt.subplots(figsize=(6,5))
scatter = ax.scatter(X_2d[:,0], X_2d[:,1], c=labels, cmap='Set1', alpha=0.85, s=100, edgecolors='white')
for i, row in enumerate(fsm_df.itertuples()):
    ax.annotate(str(row.email).split('@')[0][:8], (X_2d[i,0], X_2d[i,1]),
                fontsize=7, xytext=(4,4), textcoords='offset points')
plt.colorbar(scatter, ax=ax, label='Cluster')
ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}%)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}%)')
ax.set_title('Figure 3. Clustering of FSM-based learning trajectories (K=4)')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure3_FSM_trajectory_clustering.png", dpi=300); plt.close()
print(f'Fig 3 guardada. Varianza PC1+PC2: {sum(pca.explained_variance_ratio_)*100:.1f}%')


In [ ]:
# Figura 4 — Z por cluster
z_cluster = (z_summary_prop.merge(clusters[['email','cluster']], on=ID_COL)
             .groupby('cluster')[Z_COLS].mean())
fig, ax = plt.subplots(figsize=(7,4))
z_cluster.plot(kind='bar', ax=ax, colormap='Set2', edgecolor='white')
ax.set_ylabel('Mean proportion'); ax.set_title('Figure 4. Latent states (HMM K=4) by behavioral cluster')
ax.legend(title='Latent state', bbox_to_anchor=(1.01,1), loc='upper left')
plt.xticks(rotation=0); plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure4_Z_by_cluster.png", dpi=300); plt.close()
print('Fig 4 guardada.')


In [ ]:
# Figura 5 — Boxplot Z2 por cluster
z2_df = z_summary_prop[['email','Z2']].merge(clusters[['email','cluster']], on='email')
fig, ax = plt.subplots(figsize=(6,4))
sns.boxplot(data=z2_df, x='cluster', y='Z2', ax=ax, hue='cluster', legend=False, palette='Set2')
ax.set_title('Figure 5. Distribution of latent state Z2 by cluster')
ax.set_ylabel('Z2 proportion'); ax.set_xlabel('Behavioral cluster')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure5_Z2_by_cluster.png", dpi=300); plt.close()
print('Fig 5 guardada.')


In [ ]:
# Figura 6 — Pie modos cognitivos
pie_vals = z_summary_prop[Z_COLS].mean().values
fig, ax = plt.subplots(figsize=(5,5))
ax.pie(pie_vals, labels=Z_COLS, autopct='%1.1f%%', startangle=90,
       colors=['#5C85D6','#66BB6A','#EF5350','#FFA726'])
ax.set_title('Figure 6. Mean distribution of cognitive modes')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure6_cognitive_modes_pie.png", dpi=300); plt.close()
print('Fig 6 guardada.')


In [ ]:
# Figura 7 — Heatmap Spearman FSM vs nota_final
fsm_df['email'] = fsm_df['email'].str.strip().str.lower()
grades_df['email'] = grades_df['email'].str.strip().str.lower()
df = fsm_df.merge(grades_df[['email', GRADE_COL]], on='email', how='inner')
fsm_corr_cols = [c for c in ['NAV_prop','REC_prop','PRACT_prop','QUIZ_prop','EVAL_prop'] if c in df.columns]
corr_vals = []
for c in fsm_corr_cols:
    rho, p = spearmanr(df[c], df[GRADE_COL], nan_policy='omit')
    corr_vals.append(rho)
corr_df = pd.DataFrame({'Spearman_rho': corr_vals},
                        index=[c.replace('_prop','') for c in fsm_corr_cols])
fig, ax = plt.subplots(figsize=(4,4))
sns.heatmap(corr_df[['Spearman_rho']], annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, cbar_kws={'label':'Spearman ρ'}, ax=ax)
ax.set_title('Figure 7. Correlation FSM states vs final grade')
ax.set_ylabel('FSM state'); ax.set_xlabel('')
plt.tight_layout()
plt.savefig(f"{OUTPUTS}/Figure7_FSM_vs_Grade_Spearman.png", dpi=300); plt.close()
print('Fig 7 guardada.')
print(corr_df.round(3))
print('\nNotebook 6 completo — 7 figuras generadas.')
